# RAG Pipeline Design — Notebook Overview

This notebook demonstrates the finalized Multi-Page RAG pipeline and documents the design choices, backend selection logic, and recommended experiments/tests.

Key points:
- Backends supported: OpenAI (remote), Ollama (local), HuggingFace (local/hosted), and a TF-IDF fallback.
- Selection priority (high-level): auto -> OpenAI -> Ollama autodetect -> HuggingFace -> TF-IDF fallback.
- Rationale: provide portability (offline TF-IDF), high-quality results (OpenAI), and a local, low-latency option (Ollama) for development and CI.

What you'll see when running cells:
- Informational logs indicating which backend was chosen, e.g. 'Using OpenAI embeddings via FAISS.' or 'Falling back to TF-IDF vector store.'
- When an LLM runs, outputs include a Sources list derived from chunk metadata so answers are traceable.

Suggested experiments (in order):
1. Run the notebook with no special env vars — observe whether HuggingFace or TF-IDF is used depending on your environment.
2. Set `OPENAI_API_KEY` and re-run to see OpenAI selected (if available).
3. Start a local Ollama instance and use the CLI flag `--backend ollama` or set `OLLAMA_URL` to test local model behavior.
4. Run the chunking unit tests (see tests/) to validate deterministic behavior with seeds.

Notes:
- Cells later in this notebook implement the same helper functions used by the CLI: fetching, cleaning, randomized chunking, vector store construction, retrieval search, and RetrievalQA.

## 0 — Install dependencies

Install required packages in your environment. If you're running this notebook locally, run:

```bash
pip install -r requirements.txt
# or individually:
# pip install langchain faiss-cpu openai requests beautifulsoup4 scikit-learn numpy sentence-transformers transformers
```

If you don't want to install heavy packages, the TF-IDF fallback will still work.

In [72]:
# 1 — Imports & Configuration
import os, re, random, logging
from typing import List, Optional, Tuple, Any
import requests
from bs4 import BeautifulSoup

# Optional libraries; the notebook gracefully falls back if they are not installed.
try:
    from langchain.docstore.document import Document
    from langchain.embeddings import OpenAIEmbeddings, HuggingFaceEmbeddings
    from langchain.vectorstores import FAISS
    from langchain.chains import RetrievalQA
    from langchain.llms import OpenAI, HuggingFaceHub
    HAS_LANGCHAIN = True
except Exception as e:
    print('LangChain not available or partial:', e)
    HAS_LANGCHAIN = False

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    import numpy as np
    HAS_SKLEARN = True
except Exception as e:
    print('scikit-learn not available:', e)
    HAS_SKLEARN = False

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger()

# Chunking parameters
OVERLAP = 50
MIN_CHUNK = 400
MAX_CHUNK = 600

# Default URLs (you can change these)
URL_QC = 'https://en.wikipedia.org/wiki/Quantum_computing'
URL_QML = 'https://en.wikipedia.org/wiki/Quantum_machine_learning'


In [73]:
# 2 — Fetching & Cleaning functions
from typing import Tuple
def fetch_html(url: str) -> str:
    resp = requests.get(url, headers={"User-Agent": "multi-page-rag/1.0"})
    resp.raise_for_status()
    return resp.text

def clean_wikipedia_html(html: str) -> Tuple[str, str]:
    soup = BeautifulSoup(html, 'html.parser')
    title_tag = soup.find('h1', id='firstHeading')
    title = title_tag.get_text(strip=True) if title_tag else ''
    content = soup.find('div', id='mw-content-text') or soup.find('article') or soup
    for tag in content.find_all(['table','style','script','sup','img','aside']):
        tag.decompose()
    stop_headings = {'References','External links','See also','Further reading'}
    paragraphs = []
    for elem in content.find_all(['h2','h3','p','li']):
        if elem.name in ('h2','h3'):
            heading_text = re.sub(r"\[.*?\]", '', elem.get_text(' ', strip=True))
            if any(h in heading_text for h in stop_headings):
                break
            continue
        text = elem.get_text(' ', strip=True)
        if not text:
            continue
        text = re.sub(r"\[\d+\]", '', text)
        text = re.sub(r"\[citation needed\]", '', text, flags=re.IGNORECASE)
        text = re.sub(r"\[.*?\]", '', text)
        text = re.sub(r"\s+", ' ', text).strip()
        if len(text) > 30:
            paragraphs.append(text)
    cleaned = '\n\n'.join(paragraphs)
    return title, cleaned

# Quick preview
for url in [URL_QC, URL_QML]:
    print('Fetching and cleaning:', url)
    html = fetch_html(url)
    title, cleaned = clean_wikipedia_html(html)
    print('\nTitle:', title)
    print('\nPreview (first 400 chars):\n', cleaned[:400].replace('\n',' '))
    print('\n---\n')


Fetching and cleaning: https://en.wikipedia.org/wiki/Quantum_computing

Title: Quantum computing

Preview (first 400 chars):
 A quantum computer is a (real or theoretical) computer that uses quantum mechanical phenomena in an essential way: it exploits superposed and entangled states , and the intrinsically non-deterministic outcomes of quantum measurements , as features of its computation. Quantum computers can be viewed as sampling from quantum systems that evolve in ways classically described as operating on an enormo

---

Fetching and cleaning: https://en.wikipedia.org/wiki/Quantum_machine_learning

Title: Quantum machine learning

Preview (first 400 chars):
 Quantum machine learning (QML), pioneered by Ventura and Martinez and by Trugenberger in the late 1990s and early 2000s , is the study of quantum algorithms which solve machine learning tasks.  The most common use of the term refers to quantum algorithms for machine learning tasks which analyze classical data, sometimes calle

In [74]:
# 3 — Randomized overlapping chunking
import random
def randomized_chunks(text: str, min_size=MIN_CHUNK, max_size=MAX_CHUNK, overlap=OVERLAP, seed: Optional[int] = None):
    if seed is not None:
        random.seed(seed)
    pos = 0
    n = len(text)
    chunks = []
    while pos < n:
        size = random.randint(min_size, max_size)
        chunk = text[pos: pos + size].strip()
        if not chunk:
            break
        chunks.append(chunk)
        pos += max(1, size - overlap)
        if len(chunks) > 10000:
            break
    return chunks

# Example: chunk QC cleaned text
_, cleaned_qc = clean_wikipedia_html(fetch_html(URL_QC))
chunks_qc = randomized_chunks(cleaned_qc, seed=123)
print('QC chunks:', len(chunks_qc))
print('Example chunk length distribution: min', min(len(c) for c in chunks_qc), 'max', max(len(c) for c in chunks_qc))
print('\nFirst chunk preview:\n', chunks_qc[0][:500])


QC chunks: 103
Example chunk length distribution: min 192 max 598

First chunk preview:
 A quantum computer is a (real or theoretical) computer that uses quantum mechanical phenomena in an essential way: it exploits superposed and entangled states , and the intrinsically non-deterministic outcomes of quantum measurements , as features of its computation. Quantum computers can be viewed as sampling from quantum systems that evolve in ways classically described as operating on an enormous number of


In [75]:
# 4 — Build documents (chunks with metadata)
from dataclasses import dataclass
def build_docs_for_urls(urls: List[str], seed: Optional[int] = None):
    docs = []
    for url in urls:
        html = fetch_html(url)
        title, cleaned = clean_wikipedia_html(html)
        chunks = randomized_chunks(cleaned, seed=seed)
        for i, c in enumerate(chunks):
            if 'Document' in globals():
                docs.append(Document(page_content=c, metadata={'source': url, 'title': title, 'chunk_index': i}))
            else:
                docs.append({'page_content': c, 'metadata': {'source': url, 'title': title, 'chunk_index': i}})
    return docs

urls = [URL_QC, URL_QML]
docs = build_docs_for_urls(urls, seed=42)
print('Total docs (chunks):', len(docs))
print('Sample metadata:', docs[0].metadata if hasattr(docs[0], 'metadata') else docs[0]['metadata'])


Total docs (chunks): 182
Sample metadata: {'source': 'https://en.wikipedia.org/wiki/Quantum_computing', 'title': 'Quantum computing', 'chunk_index': 0}


In [76]:
# 5 — Build vector store (OpenAI / HuggingFace / TF-IDF fallback)
def build_vectorstore(docs: List[Any]):
    # Try OpenAI embeddings via LangChain
    if HAS_LANGCHAIN and os.getenv('OPENAI_API_KEY'):
        try:
            emb = OpenAIEmbeddings()
            store = FAISS.from_documents(docs, emb)
            return store, 'faiss-openai'
        except Exception as e:
            print('OpenAI FAISS build failed:', e)
    # Try HuggingFace via LangChain
    if HAS_LANGCHAIN:
        try:
            emb = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
            store = FAISS.from_documents(docs, emb)
            return store, 'faiss-hf'
        except Exception as e:
            print('HuggingFace FAISS build failed:', e)
    # Fallback to TF-IDF
    if HAS_SKLEARN:
        class SimpleFallbackVectorStore:
            def __init__(self, docs):
                self.docs = docs
                texts = [d.page_content if hasattr(d, 'page_content') else d['page_content'] for d in docs]
                self.tfidf = TfidfVectorizer().fit(texts + ['placeholder'])
                self.vectors = self.tfidf.transform(texts).toarray()
            def search(self, query, k=3):
                qv = self.tfidf.transform([query]).toarray()[0]
                sims = (self.vectors @ qv) / ((np.linalg.norm(self.vectors, axis=1) * (np.linalg.norm(qv) + 1e-9)) + 1e-9)
                idxs = sims.argsort()[::-1][:k]
                return [(self.docs[i], float(sims[i])) for i in idxs]
        return SimpleFallbackVectorStore(docs), 'tfidf'
    raise RuntimeError('No vector store available')

store, stype = build_vectorstore(docs)
print('Built vector store type:', stype)


2025-10-05 21:09:36,336 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Built vector store type: faiss-openai


In [77]:
# 6 — Similarity search function
import numpy as np

def search_db(vector_store, query: str, k: int = 3, filter_source: Optional[str] = None):
    if hasattr(vector_store, 'search') and not hasattr(vector_store, 'similarity_search_with_score'):
        docs_and_scores = vector_store.search(query, k * 3)
    else:
        docs_and_scores = vector_store.similarity_search_with_score(query, k * 3)
    filtered = []
    for doc, score in docs_and_scores:
        meta = doc.metadata if hasattr(doc, 'metadata') else doc.get('metadata', {})
        src = meta.get('source') if meta else None
        if filter_source and src and filter_source not in src:
            continue
        filtered.append((doc, score))
        if len(filtered) >= k:
            break
    out = []
    for doc, score in filtered:
        meta = doc.metadata if hasattr(doc, 'metadata') else doc.get('metadata', {})
        out.append({'chunk': doc.page_content, 'score': float(score), 'source': meta.get('source')})
    return out

# Test queries
queries = [
    'what are Quantum neural networks?',
    'What is the basic unit of information in quantum computing?'
]
for q in queries:
    print("\nQuery:", q)
    hits = search_db(store, q, k=3)
    for i, h in enumerate(hits, 1):
        chunk_preview = h["chunk"][:200].replace("\n", " ")
        print(f"HIT {i}: score={h['score']:.4f} source={h['source']} chunk_preview={chunk_preview}…")




Query: what are Quantum neural networks?


2025-10-05 21:10:27,740 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


HIT 1: score=0.2225 source=https://en.wikipedia.org/wiki/Quantum_machine_learning chunk_preview=extension of neural networks using photons, layered variational circuits or quantum Ising-type models .  A novel design for multi-dimensional vectors that uses circuits as convolution filters is QCNN.…
HIT 2: score=0.2589 source=https://en.wikipedia.org/wiki/Quantum_machine_learning chunk_preview=e, and, while the protocol relies on a universal quantum computer, under mild assumptions it can be embedded on contemporary quantum annealing hardware.  Quantum analogues or generalizations of classi…
HIT 3: score=0.2703 source=https://en.wikipedia.org/wiki/Quantum_machine_learning chunk_preview=doing so, the company is encouraging software developers to pursue new algorithms through a development environment with quantum capabilities. New architectures are being explored on an experimental b…

Query: What is the basic unit of information in quantum computing?


2025-10-05 21:10:28,344 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


HIT 1: score=0.2109 source=https://en.wikipedia.org/wiki/Quantum_computing chunk_preview=mental and only suitable for specialized tasks.  The basic unit of information in quantum computing, the qubit (or "quantum bit"), serves the same function as the bit in ordinary or "classical" comput…
HIT 2: score=0.2398 source=https://en.wikipedia.org/wiki/Quantum_computing chunk_preview=come from?" We should say, "well, all computers are quantum. ... Where do classical slowdowns come from?"  Just as the bit is the basic concept of classical information theory, the qubit is the fundam…
HIT 3: score=0.3022 source=https://en.wikipedia.org/wiki/Quantum_computing chunk_preview=cosmological information bound implied by the holographic principle . Skeptics like Gil Kalai doubt that quantum supremacy will ever be achieved. Physicist Mikhail Dyakonov has expressed skepticism of…


In [81]:
# 7 — RetrievalQA wrapper (LangChain if available, else simple concatenation)
def build_retrieval_qa(vector_store):
    if HAS_LANGCHAIN:
        try:
            if os.getenv('OPENAI_API_KEY'):
                llm = OpenAI(temperature=0)
            else:
                llm = HuggingFaceHub(repo_id='tiiuae/falcon-7b-instruct', model_kwargs={'temperature': 0})
            retriever = vector_store.as_retriever(search_kwargs={'k': 4})
            qa = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', retriever=retriever, return_source_documents=True)
            return qa, 'langchain'
        except Exception as e:
            print('LangChain RetrievalQA construction failed:', e)
    def simple_qa(query: str, k: int = 3, filter_source: Optional[str] = None):
        hits = search_db(vector_store, query, k=k, filter_source=filter_source)
        if not hits:
            return {'answer': "I don't know based on the provided documents.", 'sources': []}
        parts = [h['chunk'] for h in hits]
        sources = [h['source'] for h in hits if h.get('source')]
        return {'answer': '\n\n'.join(parts), 'sources': list(dict.fromkeys(sources))}
    return simple_qa, 'simple'

qa, qatype = build_retrieval_qa(store)
print('QA type:', qatype)

# Example unrestricted retrieval
for q in queries:
    print('\nQUESTION:', q)
    if qatype == 'langchain':
        try:
            res = qa.run(q)
            print('LLM answer:\n', res)
        except Exception as e:
            print('LLM run failed:', e)
    else:
        res = qa(q, k=3)
        print('Answer (concatenated):\n', res['answer'][:800])
        print('Sources:', res['sources'])


LangChain RetrievalQA construction failed: 'SimpleFallbackVectorStore' object has no attribute 'as_retriever'
QA type: simple

QUESTION: what are Quantum neural networks?
Answer (concatenated):
 This is a test page content about qubits and Quantum neural networks (QNN).

This is a test page content about qubits and Quantum neural networks (QNN).
Sources: ['https://en.wikipedia.org/wiki/Quantum_computing', 'https://en.wikipedia.org/wiki/Quantum_machine_learning']

QUESTION: What is the basic unit of information in quantum computing?
Answer (concatenated):
 This is a test page content about qubits and Quantum neural networks (QNN).

This is a test page content about qubits and Quantum neural networks (QNN).
Sources: ['https://en.wikipedia.org/wiki/Quantum_computing', 'https://en.wikipedia.org/wiki/Quantum_machine_learning']


In [82]:
# 8 — Page-specific QA wrappers and comparison
def ask_QC(query: str):
    if qatype == 'langchain':
        print('LangChain retriever — page-specific filtering not implemented in this simple demo')
        return None
    else:
        return qa(query, k=3, filter_source=URL_QC)

def ask_QML(query: str):
    if qatype == 'langchain':
        print('LangChain retriever — page-specific filtering not implemented in this simple demo')
        return None
    else:
        return qa(query, k=3, filter_source=URL_QML)

sample_q = 'what are Quantum neural networks?'
print('Unrestricted:')
print(qa(sample_q) if qatype != 'langchain' else 'See LangChain result above')
print('\nOnly QC page:')
print(ask_QC(sample_q))
print('\nOnly QML page:')
print(ask_QML(sample_q))


Unrestricted:
{'answer': 'This is a test page content about qubits and Quantum neural networks (QNN).\n\nThis is a test page content about qubits and Quantum neural networks (QNN).', 'sources': ['https://en.wikipedia.org/wiki/Quantum_computing', 'https://en.wikipedia.org/wiki/Quantum_machine_learning']}

Only QC page:
{'answer': 'This is a test page content about qubits and Quantum neural networks (QNN).', 'sources': ['https://en.wikipedia.org/wiki/Quantum_computing']}

Only QML page:
{'answer': 'This is a test page content about qubits and Quantum neural networks (QNN).', 'sources': ['https://en.wikipedia.org/wiki/Quantum_machine_learning']}


In [83]:
# 9 — Quick test-like checks (replicates key unit-test scenarios)
import sys
import types
from pprint import pprint
# Go up one level from 'notebook' to 'root'
sys.path.append(os.path.abspath(".."))

from src import rag_pipeline as rp

# 1) Stub fetch_html for deterministic content
def stub_fetch_html(url):
    return """<html><h1 id='firstHeading'>Test Page</h1><div id='mw-content-text'><p>This is a test page content about qubits and Quantum neural networks (QNN).</p></div></html>"""

rp.fetch_html = lambda url: stub_fetch_html(url)
print('fetch_html stubbed')

# 2) Simulate no LangChain available and ensure build_from_urls produces document-like objects
orig_doc = getattr(rp, 'Document', None)
orig_has_lang = getattr(rp, 'HAS_LANGCHAIN', True)
try:
    rp.Document = None
    rp.HAS_LANGCHAIN = False
    docs, store = rp.build_from_urls(['https://en.wikipedia.org/wiki/Quantum_machine_learning','https://en.wikipedia.org/wiki/Quantum_computing'], seed=1)
    print('\nbuild_from_urls returned', len(docs), 'docs')
    first = docs[0]
    # Support both dict-style and object-style docs
    if isinstance(first, dict):
        wrapped = types.SimpleNamespace(page_content=first.get('page_content'), metadata=first.get('metadata'))
    else:
        wrapped = first
    print('Has page_content attribute:', hasattr(wrapped, 'page_content'))
    print('Has metadata attribute:', hasattr(wrapped, 'metadata'))
    pprint(getattr(wrapped, 'metadata', None))

    # 3) Build vectorstore and ensure TF-IDF fallback is returned
    store2, stype = rp.build_vectorstore(docs, backend='auto')
    print('\nbuild_vectorstore type:', stype)
    if hasattr(store2, 'search'):
        hits = store2.search('qubits', k=1)
        print('TF-IDF store search returned', len(hits), 'hit(s)')
    else:
        print('store2 has no search')

    # 4) Use search_db (repo helper) and test filter_source behavior
    hits_global = rp.search_db(store2, 'Quantum neural networks', k=3)
    print('\nsearch_db global hits:')
    pprint(hits_global)

    hits_filtered = rp.search_db(store2, 'Quantum neural networks', k=3, filter_source='Quantum_machine_learning')
    print('\nsearch_db filtered hits:')
    pprint(hits_filtered)

    # 5) Make page-specific askers and call them
    askers = rp.make_page_specific_askers(store2, ['https://en.wikipedia.org/wiki/Quantum_machine_learning'])
    print('\nmake_page_specific_askers keys:', list(askers.keys()))
    if askers:
        slug = list(askers.keys())[0]
        print('Calling asker for slug:', slug)
        resp = askers[slug]('what is a qubit?', k=2)
        pprint(resp)

finally:
    # restore
    rp.Document = orig_doc
    rp.HAS_LANGCHAIN = orig_has_lang

print('\nObservations:')
print('- search_db returns global and filtered hits; page-specific askers exist and return evidence with sources')


2025-10-05 21:11:05,072 INFO Fetching https://en.wikipedia.org/wiki/Quantum_machine_learning
2025-10-05 21:11:05,075 INFO Created 1 chunks for Quantum_machine_learning
2025-10-05 21:11:05,076 INFO Fetching https://en.wikipedia.org/wiki/Quantum_computing
2025-10-05 21:11:05,076 INFO Created 1 chunks for Quantum_computing
2025-10-05 21:11:05,076 WARNING ⚠️ Falling back to TF-IDF vector store.
2025-10-05 21:11:05,076 INFO Built vector store: tfidf
2025-10-05 21:11:05,076 WARNING ⚠️ Falling back to TF-IDF vector store.
2025-10-05 21:11:05,084 INFO search_db: q=Quantum neural networks k=3 returned=2 filter=None
2025-10-05 21:11:05,086 INFO search_db: q=Quantum neural networks k=3 returned=1 filter=Quantum_machine_learning
2025-10-05 21:11:05,088 INFO search_db: q=what is a qubit? k=2 returned=1 filter=https://en.wikipedia.org/wiki/Quantum_machine_learning


fetch_html stubbed

build_from_urls returned 2 docs
Has page_content attribute: True
Has metadata attribute: True
{'chunk_index': 0,
 'source': 'https://en.wikipedia.org/wiki/Quantum_machine_learning',
 'title': 'Test Page'}

build_vectorstore type: tfidf
TF-IDF store search returned 1 hit(s)

search_db global hits:
[{'chunk': 'This is a test page content about qubits and Quantum neural '
           'networks (QNN).',
  'id': None,
  'score': 0.4999999989999999,
  'source': 'https://en.wikipedia.org/wiki/Quantum_computing'},
 {'chunk': 'This is a test page content about qubits and Quantum neural '
           'networks (QNN).',
  'id': None,
  'score': 0.4999999989999999,
  'source': 'https://en.wikipedia.org/wiki/Quantum_machine_learning'}]

search_db filtered hits:
[{'chunk': 'This is a test page content about qubits and Quantum neural '
           'networks (QNN).',
  'id': None,
  'score': 0.4999999989999999,
  'source': 'https://en.wikipedia.org/wiki/Quantum_machine_learning'}]

ma